# OVENTI · Calibre — Electricity application

Hourly point forecasts and 90% conformal prediction intervals.
Point errors belong to the forecaster, not to the calibrator.
See [README.md](README.md) for setup, the protocol and calibration scope.

## Setup

In [ ]:
import json
import sys
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src/calibre").is_dir() and (p / "pyproject.toml").exists()
)
# bench is a folder of scripts, not a package.
for directory in (ROOT / "bench", ROOT / "notebooks"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

from electricity_application import OVENTI, ExperimentReporter, describe_data  # noqa: E402

import lstnet  # noqa: E402
from electricity_bootstrap import bootstrap_metrics, loss_diagnostics  # noqa: E402
from electricity_conformal import (  # noqa: E402
    INTERVAL_METRICS,
    POINT_METRICS,
    Config,
    chronological_splits,
    identify_sarima,
    run,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 85)

In [ ]:
TARGET_COLUMN = 0
CONFIG = Config(
    n_split=5,
    train_size=30 * 24,
    calibration_size=7 * 24,
    test_size=7 * 24,
    level=0.90,
    n_lags=24,
    period=24,
    moving_window=24,
    sarima_d=1,
    sarima_method="CSS-ML",
    seasonal_order=(1, 0, 0),
    booster_iterations=200,
    booster_depth=5,
    ogd_eta=0.001,
    ogd_l2=0.001,
    aci_gamma=0.005,
    pid_rate=0.05,
    pid_integral=0.10,
    spci_n_lags=24,
    spci_min_train_samples=20,
    spci_n_estimators=100,
    spci_max_depth=5,
    spci_min_samples_leaf=5,
    seed=42,
)
SARIMA_MAX_P, SARIMA_MAX_Q = 2, 2
BOOTSTRAP = {
    "n_bootstrap": 10_000,
    "block_length": 48,
    "period": 24,
    "confidence": 0.95,
    "seed": 42,
}
BLOCK_LENGTHS = (24, 48, 72, 168, 336)
WRITE_RESULTS = True
CREATED_AT = datetime.now(UTC)
OUTPUT = ROOT / "results/electricity_notebook" / CREATED_AT.strftime("%Y%m%dT%H%M%S-%fZ")
reporter = ExperimentReporter(level=CONFIG.level, output=OUTPUT if WRITE_RESULTS else None)
display(pd.Series(asdict(CONFIG), name="Configuration"))
display(pd.Series(BOOTSTRAP, name="Temporal bootstrap"))

In [ ]:
sns.set_theme(style="whitegrid", context="notebook", palette=list(OVENTI.values()))
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 170,
        "axes.titleweight": "bold",
        "axes.titlecolor": OVENTI["medium"],
        "axes.labelcolor": OVENTI["dark"],
        "axes.edgecolor": OVENTI["medium"],
        "font.size": 10,
    }
)

## Data quality and exploration

Load the already-downloaded LSTNet matrix. Anonymous clients have no source dates
or verified physical units; plots use observation indexes and the published scale.

In [ ]:
data = lstnet.load(ROOT / "data/electricity")
y = data.panel.values[TARGET_COLUMN]
df = pd.DataFrame(data.panel.values.T, columns=data.panel.series)
df.index.name = "observation"
TARGET = df.columns[TARGET_COLUMN]
DISPLAY_COLUMNS = list(df.columns[:5])
print(f"Target: {TARGET}; {len(y):,} observations.")
display(df.loc[:, DISPLAY_COLUMNS].head())

In [ ]:
quality, descriptive = describe_data(df)
display(quality.to_frame())
print("Descriptive statistics: first five clients; full table available in CSV.")
with pd.option_context("display.max_rows", 5, "display.max_columns", None):
    display(descriptive.loc[DISPLAY_COLUMNS])
if WRITE_RESULTS:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    quality.to_csv(OUTPUT / "data_quality.csv")
    descriptive.to_csv(OUTPUT / "descriptive_all_clients.csv")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 6), layout="constrained")
axes[0].plot(df.index, y, color=OVENTI["light"], lw=0.65)
axes[0].set(title=f"OVENTI | {TARGET} — complete trajectory", ylabel="Published load scale")
zoom = df.iloc[-14 * 24 :]
axes[1].plot(zoom.index, zoom[TARGET], color=OVENTI["teal"], lw=1)
axes[1].set(title="Final 336 observations", ylabel="Published load scale")
for ax in axes:
    ax.set_xlabel("Observation index (hourly cadence; no source dates)")
reporter.save(fig, "01_trajectory")
plt.show()

fig, axes = plt.subplots(1, 5, figsize=(17, 3), layout="constrained")
for ax, name in zip(axes, DISPLAY_COLUMNS, strict=True):
    sns.histplot(df[name], bins=45, ax=ax, color=OVENTI["light"])
    ax.axvline(df[name].mean(), color=OVENTI["dark"], ls="--", label="Mean")
    ax.axvline(df[name].median(), color=OVENTI["teal"], ls=":", label="Median")
    ax.set(title=name, xlabel="Published load scale", ylabel="Observations")
axes[-1].legend(fontsize=8)
fig.suptitle("OVENTI | First five clients — marginal distributions")
reporter.save(fig, "02_client_histograms")
plt.show()

## Training-only SARIMA identification

ACF/PACF are descriptive. The first training block's AIC selects the shared order;
see the protocol in [README.md](README.md).

In [ ]:
SPLITS = chronological_splits(len(y), CONFIG)
split_table = pd.DataFrame(
    [
        {
            "fold": s.fold,
            "train_from": s.train_start,
            "train_to": s.train_end - 1,
            "calibration_from": s.train_end,
            "calibration_to": s.test_start - 1,
            "test_from": s.test_start,
            "test_to": s.test_end - 1,
            "H": s.test_end - s.test_start,
        }
        for s in SPLITS
    ]
)
display(split_table)
first = SPLITS[0]
training = y[first.train_start : first.train_end]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), layout="constrained")
for row, (label, z) in enumerate(
    [("Raw training target", training), ("First difference", np.diff(training, n=CONFIG.sarima_d))]
):
    lags = min(4 * CONFIG.period, len(z) // 2 - 1)
    plot_acf(z, lags=lags, zero=False, ax=axes[row, 0], color=OVENTI["light"])
    plot_pacf(z, lags=lags, zero=False, method="ywm", ax=axes[row, 1], color=OVENTI["teal"])
    axes[row, 0].set_title(f"{label} — ACF")
    axes[row, 1].set_title(f"{label} — PACF")
reporter.save(fig, "03_training_acf_pacf")
plt.show()

SARIMA_ORDER, sarima_identification = identify_sarima(
    training, CONFIG, max_p=SARIMA_MAX_P, max_q=SARIMA_MAX_Q
)
display(sarima_identification)
print("Training-AIC selected order:", SARIMA_ORDER)

## Point results — once per forecaster

For $H$ test targets, $e_t=\widehat Y_t-Y_t$:
$RMSE=\sqrt{H^{-1}\sum_t e_t^2}$, $MAE=H^{-1}\sum_t|e_t|$,
and $\mathrm{bias}=H^{-1}\sum_t e_t$. Here $H$ is the test-block length, not forecast lead.

In [ ]:
point_folds, interval_folds, losses = run(y, SPLITS, CONFIG, SARIMA_ORDER)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(point_folds)

## Interval results — four methods per forecaster

Coverage is endpoint membership; width is $U-L$. These metrics use
finite-threshold cells; `ready` reports their share. Width and interval score
assume ordered endpoints ([semantics](../docs/semantics.md)).

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(interval_folds)

### Mean-fold metrics with temporal bootstrap

Bars remain the observed arithmetic means across folds: **mean fold RMSE, not pooled RMSE**.
Caps are pointwise 95% empirical intervals for these metrics, distinct from the
nominal **90% prediction coverage**. Shared circular, phase-aligned 48-hour blocks
resample the full 840-hour loss record, never folds independently; forecasts stay fixed.
These are conditional sensitivity summaries, not guaranteed confidence intervals.
See the single [bootstrap protocol](README.md#temporal-bootstrap-protocol) for assumptions.

In [ ]:
bootstrap = bootstrap_metrics(losses, **BOOTSTRAP)
SUMMARY_COLUMNS = ["family", "metric", "estimate", "lower", "upper", "status"]
print("Point metric intervals — once per forecaster")
display(bootstrap.point_summary[SUMMARY_COLUMNS])
print("Interval metric intervals — per forecaster and calibrator")
with pd.option_context("display.max_rows", None):
    display(bootstrap.interval_summary[["method", *SUMMARY_COLUMNS]])

In [ ]:
metrics_figure = reporter.plot_metrics(
    point_folds,
    interval_folds,
    POINT_METRICS,
    INTERVAL_METRICS,
    point_summary=bootstrap.point_summary,
    interval_summary=bootstrap.interval_summary,
    confidence=BOOTSTRAP["confidence"],
)
plt.show()

## Loss dependence and block-length sensitivity

Diagnostics describe squared, absolute and signed errors separately from coverage and width.
They show global-mean-biased ACF at lags 24 and 168, raw and after positional-phase
centering, with statuses and pair counts. Centering is diagnostic only; losses are not transformed.
The sensitivity scan uses 24, 48, 72, 168 and 336-hour blocks, with the same seed and settings.
Full diagnostics, endpoint Monte Carlo brackets and bootstrap-bias diagnostics are exported;
Monte Carlo brackets measure simulation noise, **not** data uncertainty ([protocol](README.md#temporal-bootstrap-protocol)).

In [ ]:
point_diagnostics, interval_diagnostics = loss_diagnostics(losses, period=BOOTSTRAP["period"])
with pd.option_context("display.max_rows", 12, "display.max_columns", None):
    print("Point loss diagnostics")
    display(point_diagnostics)
    print("Interval loss diagnostics")
    display(interval_diagnostics)

In [ ]:
bootstrap_results = {
    length: (
        bootstrap
        if length == BOOTSTRAP["block_length"]
        else bootstrap_metrics(losses, **{**BOOTSTRAP, "block_length": length})
    )
    for length in BLOCK_LENGTHS
}
point_bootstrap = pd.concat(
    [r.point_summary for r in bootstrap_results.values()], ignore_index=True
)
interval_bootstrap = pd.concat(
    [r.interval_summary for r in bootstrap_results.values()], ignore_index=True
)
print("Point metric endpoints by block length (hours)")
display(
    point_bootstrap.pivot(
        index=["family", "metric"], columns="block_length", values=["lower", "upper"]
    )
)
print("Interval metric endpoints by block length (hours)")
with pd.option_context("display.max_rows", 12):
    display(
        interval_bootstrap.pivot(
            index=["family", "method", "metric"], columns="block_length", values=["lower", "upper"]
        )
    )

## Local exports

In [ ]:
if WRITE_RESULTS:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    for name, table in {
        "point_metrics_by_fold": point_folds,
        "interval_metrics_by_fold": interval_folds,
        "point_bootstrap_intervals_primary": bootstrap.point_summary,
        "interval_bootstrap_intervals_primary": bootstrap.interval_summary,
        "point_bootstrap_intervals": point_bootstrap,
        "interval_bootstrap_intervals": interval_bootstrap,
        "point_loss_diagnostics": point_diagnostics,
        "interval_loss_diagnostics": interval_diagnostics,
        "chronological_splits": split_table,
        "sarima_training_identification": sarima_identification,
    }.items():
        table.to_csv(OUTPUT / f"{name}.csv", index=False)
    reference = bootstrap.point_summary.iloc[0]
    point_axes = ["block_length", "replicate", "family", "point_metric"]
    interval_axes = ["block_length", "replicate", "family", "method", "interval_metric"]
    np.savez_compressed(
        OUTPUT / "bootstrap_draws.npz",
        point_draws=np.stack([r.point_draws for r in bootstrap_results.values()]),
        interval_draws=np.stack([r.interval_draws for r in bootstrap_results.values()]),
        point_draw_axes=np.asarray(point_axes),
        interval_draw_axes=np.asarray(interval_axes),
        families=np.asarray(losses.families),
        methods=np.asarray(losses.methods),
        steps=losses.steps,
        fold_sizes=losses.fold_sizes,
        point_metrics=np.asarray(POINT_METRICS),
        interval_metrics=np.asarray(INTERVAL_METRICS),
        point_losses=losses.point,
        interval_losses=losses.interval,
        ready=losses.ready,
        point_loss_axes=np.asarray(["step", "family", "point_loss"]),
        interval_loss_axes=np.asarray(["step", "family", "method", "interval_metric"]),
        ready_axes=np.asarray(["step", "family", "method"]),
        point_loss_metrics=np.asarray(["squared_error", "absolute_error", "error"]),
        block_lengths=np.asarray(BLOCK_LENGTHS),
        primary_block_length=BOOTSTRAP["block_length"],
        n_bootstrap=BOOTSTRAP["n_bootstrap"],
        period=BOOTSTRAP["period"],
        confidence=BOOTSTRAP["confidence"],
        seed=BOOTSTRAP["seed"],
        lower_rank=int(reference.lower_rank),
        upper_rank=int(reference.upper_rank),
        mc_confidence=float(reference.mc_confidence),
        block_lengths_in_record=np.asarray(
            [r.point_summary.block_lengths_in_record.iloc[0] for r in bootstrap_results.values()]
        ),
        blocks_per_replicate=np.asarray(
            [r.point_summary.blocks_per_replicate.iloc[0] for r in bootstrap_results.values()]
        ),
    )
    source = lstnet.SOURCES["electricity"]
    manifest = {
        "created_at_utc": CREATED_AT.isoformat(),
        "source_url": source.url,
        "source_sha256": source.sha256,
        "target": TARGET,
        "configuration": asdict(CONFIG),
        "sarima_order_training_aic": SARIMA_ORDER,
        "forecasters": np.asarray(losses.families).tolist(),
        "methods": np.asarray(losses.methods).tolist(),
        "chart": "Observed arithmetic fold means with pointwise bootstrap caps; not pooled RMSE",
        "interval_evaluation": "Standard compare metrics on finite-threshold cells; see ready",
        "bootstrap": {
            **BOOTSTRAP,
            "block_lengths": list(BLOCK_LENGTHS),
            "scope": "Conditional on fixed recorded losses; no refit or guaranteed coverage",
            "sampling": "Shared paired circular phase-aligned blocks over the whole test record",
            "statistic": "Repartition each resample into original fold slots; average fold metrics",
            "rng": "numpy.default_rng / PCG64",
            "lower_rank": int(reference.lower_rank),
            "upper_rank": int(reference.upper_rank),
            "mc_confidence": float(reference.mc_confidence),
            "mc_scope": "Endpoint brackets measure simulation noise, not data uncertainty",
            "invalid_source": "No hours/draws dropped; unready interval pairs have no metric CI",
            "point_draw_axes": point_axes,
            "interval_draw_axes": interval_axes,
            "point_metrics": list(POINT_METRICS),
            "interval_metrics": list(INTERVAL_METRICS),
            "n_test": len(losses.steps),
            "fold_sizes": np.asarray(losses.fold_sizes).tolist(),
        },
        "loss_diagnostic_lags": [BOOTSTRAP["period"], 7 * BOOTSTRAP["period"]],
    }
    (OUTPUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("Local tables, figures and bootstrap archive written.")